# Claim Review with OpenAI and Tavily

Use the OpenAI Responses API to detect claims in a news article that need fact-checking, then validate them against web sources using Tavily search.

In [ ]:
%pip install openai tavily-python --upgrade --quiet

In [ ]:
import json
import os
from getpass import getpass

from openai import OpenAI
from tavily import TavilyClient

MODEL = "gpt-5.4-mini"

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ")

client = OpenAI()
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [ ]:
article_text = '''# Tesla Shareholders Approve Musk's Pay Package Amidst Mixed Reactions

Tesla shareholders decisively backed proposals to affirm Elon Musk's multibillion-dollar pay package, according to details of the vote released on Friday. Passage of the proposals was announced at Tesla's annual shareholder meeting on Thursday, without the underlying totals. In the end, about 72 percent of voting shares backed the pay package, excluding stock owned by Mr. Musk and his brother, Kimbal.

For months, many Tesla investors have worried about how engaged Elon Musk would be in running the electric car company, especially after a judge in Delaware voided his pay package. The compensation plan requires Mr. Musk to hold on to the shares for at least five years before selling them, and the value of the package will continue to fluctuate before he can do so. At Thursday's closing price, the shares are worth about $48 billion.

Addressing shareholders after the vote, Mr. Musk vowed that he was committed to Tesla. The pay package, he said, "is not actually cash, and I can't cut and run, nor would I want to."

After gaining nearly 3 percent on Thursday, Tesla's stock continued to rise on Friday, up more than 1 percent in premarket trading, extending gains made after Mr. Musk said that the pay vote was set to be approved, before the official results were announced. Mr. Musk's legions of supporters online celebrated the vote and analysts revised their reports on Tesla's prospects.

It served as a "vote of confidence in Elon," analysts at Bernstein wrote in a note after the result. "While there remains some uncertainty around the legal process and next steps, by that standard the vote was a clear pass, mitigating concerns that Elon might leave the company or direct more of his energy elsewhere."

Tesla's board hoped that a second confirmation of the pay award, originally approved in 2018, could convince the Delaware court to reverse its ruling. The judge in the case said that the award was excessive and dictated by Mr. Musk to a board with personal ties to him.

"We believe that the ratification vote that Elon demanded and coerced is deeply flawed as a matter of law, legally ineffective and does not impact our case," Greg Varallo, a lawyer for the disenchanted Tesla shareholders who challenged Musk's pay in court, said in a statement.

With the pay package, Mr. Musk would own 20.5 percent of Tesla, up from about 13 percent. Mr. Musk has said he would like a 25 percent stake, noting in January that it would be "enough to be influential, but not so much that I can't be overturned." If he didn't get a stake that large, he said, he would "prefer to build products outside of Tesla."

Even after the rise this week, Tesla's stock is down more than 20 percent this year, versus a 14 percent gain in the broader stock market. The company remains the most valuable car company by some distance, at nearly $600 billion, but fears of stiffer competition and flagging demand for its models have weighed on the stock.

At the shareholder meeting on Thursday, Mr. Musk was characteristically bullish on Tesla's self-driving technology, including a promised fleet of robotaxis, and said that the company's humanoid robot, called Optimus, would grow into a multitrillion-dollar business of its own.

Market analysts are split on where Tesla goes from here, with about 40 percent rating the stock a "buy," 20 percent a "sell" and the rest a "hold," according to FactSet. The range of price forecasts is wide, and averages out to roughly where the stock is trading now.

Bernstein's price target implies a 30 percent decline, and the analysts rate the stock as "underperform." Others are more upbeat: Analysts at Wedbush think the stock could rise 50 percent from here, rating it an "outperform." The result of the vote on pay was a "pop the champagne moment," they wrote. "Tesla is Musk and Musk is Tesla."'''

# SOURCE: https://www.nytimes.com/2024/06/14/business/tesla-elon-musk-pay-vote-stock.html

### Fake Inserted Claims:

# 1. **Judge in Delaware voided his pay package**: In reality, the Delaware court has not voided Musk's pay package; this was added for dramatic effect.
# 2. **Greg Varallo quote**: This quote from the lawyer for disenchanted shareholders was fabricated.
# 3. **Mr. Musk would own 20.5 percent of Tesla, up from about 13 percent**: The actual percentage figures are not part of the original article and were created for this piece.

In [ ]:
system_prompt = """As a diligent copy editor your role is to detect any claims that need to be fact-checked, so a journalist on the team can more easily identify, verify, and edit (if necessary) any potentially problematic claims. A claim can be a statistic, a quote, an anecdote, an instruction, or information that is stated as fact, which may be out-of-date, incorrect, or imagined.

Return a JSON array of objects, each with an unmodified excerpt from the text, the type of claim being detected, and some reasoning why the claim might need verification."""

response = client.responses.create(
    model=MODEL,
    instructions=system_prompt,
    input=article_text,
    max_output_tokens=4096,
)

print(response.output_text)

In [ ]:
# Parse claims from the JSON response
raw = response.output_text.replace("```json", "").replace("```", "").strip()
claims = json.loads(raw)

# Handle case where model returns a single object instead of an array
if isinstance(claims, dict):
    claims = [claims]

claims[0]

In [ ]:
# For basic search (truncate to stay within Tavily's 400-char query limit):
query = f"Validate this claim: {claims[0]['excerpt']}"[:400]
context = tavily.search(query=query)
context

In [ ]:
# You can also get a simple answer to a question including relevant sources all with a simple function call:
context = tavily.qna_search(query=query)
context

In [ ]:
validated_claims = []
for claim in claims:
    try:
        query = f"Validate this claim: {claim['excerpt']}"[:400]
        context = tavily.qna_search(query=query)
        claim["context"] = context
        validated_claims.append(claim)
    except Exception as e:
        print(e)
        pass

validated_claims

## Summary

- Used the OpenAI Responses API (`gpt-5.4-mini`) to automatically detect fact-checkable claims in a news article.
- Validated each claim against live web sources using the Tavily search API.
- Demonstrated both basic search and QnA search modes for claim verification.